# Session 12. Harness engineering: how industrial agents are built

**The model is a few percent of the product. Today: everything around it.**

- inside Claude Code: one loop, five compression layers, permission walls
- three techniques run live on a scripted model, each portable to your agent
- practice: read a real extracted system prompt and steal five techniques

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## What you already built

**A harness is the course so far, shipped as one product.**

- loop (s1), graph (s2), middleware (s3), compression (s4)
- interrupts (s6), traces (s7): every piece had its session
- industrial agents are the same pieces, hardened and layered
- today we read the blueprints of the shipped ones

## Thin model, thick harness

**Claude Code: about 1.6% of the code decides. The rest executes.**

- one main while-loop, `queryLoop`: assemble context, call model, dispatch tools
- the session-1 ReAct shape at industrial grade (arXiv 2604.14228, Dive into Claude Code)
- subagents: one nesting level, sidechain transcripts, only summaries return

## Five compression layers

**Before every model call, the context earns its place.**

- per-message budgets, history snip, microcompact, context collapse, full auto-compact
- cheapest layer first; the full auto-compact is the last resort
- layer names as of August, from arXiv 2604.14228v2

## The minusx teardown

**One main loop over a flat message list, and that is deliberate.**

- minusx logged Claude Code's wire traffic and found no graph
- orchestration refused on purpose: the complexity lives in prompts and tools
- the write-up: https://minusx.ai/blog/decoding-claude-code/ — open it after class

**The model keeps its own todo list. There is no planner model.**

- the list is rewritten whole by a tool call, visible in every trace
- more than half of all model calls go to the cheap model
- codebase search is agentic ripgrep, not a vector index going stale

## Extracted system prompts

**Not one blob: 500+ conditional fragments, assembled per environment.**

- flags, OS and git state decide which fragments ship on each call
- utility prompts run beside the loop: compaction, session titles, bash security checks
- repos: Piebald-AI/claude-code-system-prompts, newest release folder; x1xhlol mirrors it
- both repos restructure weekly — the exact file is announced in class

**Steering is engineered: explicit algorithms with decision points, not a soup of Dos and Don'ts.**

- capitalized markers flag the rules the model must not trade away
- "One word answers are best" — Piebald-AI, claude-code-system-prompts
- "NEVER commit changes unless the user explicitly asks" — same repo
- `<system-reminder>` insertions: state recaps injected per call, demo 2 today
- good/bad example pairs disambiguate forks a rule alone cannot

## The open-systems tour

**Five harnesses, one idea each. Steal by name.**

- OpenHands: append-only event stream plus a pluggable history condenser
- Aider: repo map from tree-sitter tags ranked by PageRank, precomputed context
- Cline: plan/act split — plan mode reads and discusses, act mode gains write tools

**Two more, and one lesson.**

- gemini-cli: double checkpointing — file snapshots plus conversation save/resume
- opencode: LSP diagnostics fed back to the model after every edit
- five products, one lesson: the loop is simple, the walls are engineered

## The permission model

**Deny rules override allow rules. Remember that sentence.**

- graduated modes, from ask-every-time to auto within guardrails
- in auto mode a separate classifier reviews each command first
- one card today; session 15 files it among the defense layers — the one that holds even when the model is fooled

## Today's model is a script

**Harness behavior is model-independent, so today the model is fake.**

- the session-11 test trick: `GenericFakeChatModel` replaying scripted replies
- zero tokens, full determinism, in class and offline
- the cheap-model rule at its limit: the cheapest model is no model
- the config cell at the top stays for the practice half; no demo calls it

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedChatModel(GenericFakeChatModel):
    """Replays a message script; the session-11 trick as a lecture driver."""

    def bind_tools(self, tools, **kwargs):
        return self  # the script already carries the tool calls


probe = ScriptedChatModel(messages=iter([AIMessage("Scripted reply, zero tokens.")]))
print(probe.invoke("anything at all").content)  # same answer every run, no key

## The task list in state

**The teardown's todo list is first-class in your own framework.**

- `TodoListMiddleware` registers one tool, `write_todos`, and one state key, `todos`
- the model rewrites the whole list each call; statuses tell the story
- watch for the `ToolMessage` confirming every rewrite

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

PLAN = [
    {"content": "Pack the kitchen into labelled boxes", "status": "in_progress"},
    {"content": "Book a van for Saturday morning", "status": "pending"},
    {"content": "Defrost the freezer overnight", "status": "pending"},
]
LATER = [dict(PLAN[0], status="completed"),
         dict(PLAN[1], status="in_progress"), PLAN[2]]

script = iter([  # two whole-list rewrites, then a closing text reply
    AIMessage("", tool_calls=[
        {"name": "write_todos", "args": {"todos": PLAN}, "id": "t1"}]),
    AIMessage("", tool_calls=[
        {"name": "write_todos", "args": {"todos": LATER}, "id": "t2"}]),
    AIMessage("Kitchen packed; the van booking is next."),
])

mover = create_agent(ScriptedChatModel(messages=script), tools=[],
                     middleware=[TodoListMiddleware()])
moved = mover.invoke(
    {"messages": [{"role": "user", "content": "Plan my move and start packing."}]},
    config={"recursion_limit": 10},
)

for todo in moved["todos"]:  # the task list is agent state, not chat text
    print(f"{todo['status']:12} {todo['content']}")
for message in moved["messages"]:  # the ToolMessage rows confirm each rewrite
    print("  ", type(message).__name__, "-", message.content[:64])

In [ ]:
desc = TodoListMiddleware().tool_description  # the framework ships this prompt
print(len(desc), "chars of prompt behind one tool")
for line in desc.splitlines():
    if "IMPORTANT" in line:  # the lecture's steering markers, in the wild
        print(line.strip())

print(mover.get_graph().draw_mermaid())

**`todos` is state, not chat text. That is the point.**

- statuses pending, in_progress, completed; keep one task in progress
- the whole list is rewritten per call: no diff protocol to get wrong
- on your live agent this is one middleware line in `create_agent`
- the graph grew an `after_model` node: session 3's hooks are graph nodes underneath

## The system-reminder

**What the model sees is not what history stores.**

- Claude Code appends `<system-reminder>` lines: state recaps, freshly relevant rules
- injected per call, never persisted, so the thread stays clean
- we rebuild the trick with `wrap_model_call` from session 3

In [ ]:
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import SystemMessage

SEEN = []  # the exact request, captured on its way to the provider


class RecordingModel(ScriptedChatModel):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        SEEN[:] = messages  # keep a copy; the run continues unchanged
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


REMINDER = SystemMessage(
    "<system-reminder>Boxes packed: 7 of 12. Van still not booked.</system-reminder>"
)


@wrap_model_call  # every call gets the recap; the state never does
def remind(request, handler):
    return handler(request.override(messages=[*request.messages, REMINDER]))


scripted = RecordingModel(messages=iter([AIMessage("Book the van first, today.")]))
reminded = create_agent(scripted, tools=[], middleware=[remind])
state = reminded.invoke(
    {"messages": [{"role": "user", "content": "What should I do next?"}]},
    config={"recursion_limit": 5},
)

print("the model saw:")
for message in SEEN:
    print("  ", type(message).__name__, "-", message.content[:64])
print("state stores:")
for message in state["messages"]:
    print("  ", type(message).__name__, "-", message.content[:64])

**Transient injection: seen once, stored never.**

- the recap rode on the request; state holds only the user turn and the reply
- `request.override(messages=...)` is the API; direct assignment is deprecated and warns
- every call can carry fresh state without the thread ever growing

## Compression as middleware

**Session 4's summarization, packaged the way industry runs it.**

- `SummarizationMiddleware` inspects the history before every model call
- `trigger` decides when it fires; `keep` decides what survives verbatim
- the summarizer is itself a model call, with its own utility prompt

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage

history = [  # nine turns of moving-day progress, scripted verbatim
    HumanMessage("Moving day is Saturday. Where do I start?"),
    AIMessage("Start with the kitchen: plates, glasses, pans."),
    HumanMessage("Kitchen done, four boxes. Next?"),
    AIMessage("Books, then the bedroom closet."),
    HumanMessage("Books packed, closet half done. The van?"),
    AIMessage("Book the van for Saturday, the 9am slot."),
    HumanMessage("Van booked. What about the freezer?"),
    AIMessage("Defrost it Friday night, towels on the floor."),
    HumanMessage("Freezer sorted. Labels?"),
]

summarizer = ScriptedChatModel(messages=iter([AIMessage(
    "Kitchen and books packed, seven boxes; van booked Saturday 9am; "
    "freezer defrosts Friday."
)]))
main = ScriptedChatModel(messages=iter([AIMessage(
    "Label each box with its room; only the bathroom shelf is left."
)]))

compact = create_agent(main, tools=[], middleware=[SummarizationMiddleware(
    model=summarizer, trigger=("messages", 6), keep=("messages", 2),
)])

question = HumanMessage("Anything I forgot?")  # the turn that tips the trigger
out = compact.invoke({"messages": [*history, question]},
                     config={"recursion_limit": 5})

print("messages in:", len(history) + 1, "| messages out:", len(out["messages"]))
for message in out["messages"]:
    print("  ", type(message).__name__, "-", message.content[:72])

In [ ]:
shipped = SummarizationMiddleware(model=summarizer, trigger=("messages", 6))
print(shipped.summary_prompt[:160], "...")  # a utility prompt, shipped in the framework

try:  # fraction triggers must know the window size; ours has none
    SummarizationMiddleware(model=summarizer, trigger=("fraction", 0.8))
except ValueError as error:
    print("ValueError:", str(error)[:120])

**Ten messages in, four out. Nothing else changed.**

- the old turns became one `HumanMessage` starting "Here is a summary..."
- summarization is a model call: on a live agent you pay for it
- scripted or profile-less models: trigger by `("messages", N)` or `("tokens", N)`
- `("fraction", ...)` needs a model profile — you just saw the `ValueError`

## Cheap-model routing

**minusx's biggest number — over half of calls — is one line of routing.**

```python
@wrap_model_call
def route(request, handler):
    if is_simple(request):  # your test: no pending tools, short context
        return handler(request.override(model=cheap_model))
    return handler(request)
```

Not executed today: the scripted lecture already runs on the cheapest model there is.

## Practice

**Read a real harness prompt, in pairs, with the file open.**

1. open Piebald-AI/claude-code-system-prompts, newest release folder (x1xhlol mirrors it)
2. find five distinct techniques; note the file and the line for each
3. classify each one: marker, reminder, example pair, utility prompt, or algorithm

**Transfer two techniques into your own project agent.**

4. pick two: `TodoListMiddleware`, `SummarizationMiddleware`, or a `wrap_model_call` reminder
5. run the same dialogue before and after; export both traces from Langfuse
6. the traces must show the change: a new tool call, a shorter context, an injected line
7. free tiers beware: `write_todos` turns and summary calls cost tokens

**Required artifact: `runs/session-12.md`, committed.**

- the five techniques, each with its file pointer
- the two you transferred, and why those two
- both trace exports, before and after, linked or attached
- stretch: route simple turns to your cheap model via `request.override(model=...)`, then measure the share of calls it takes in traces

## The harness checklist

Ten questions for your project agent. Keep the list next to the keyboard.

```text
 1. One readable main loop — could you draw it from memory?
 2. Subagents return summaries only, never full transcripts.
 3. Compression is layered and runs before calls, not after crashes.
 4. Todo discipline: the list lives in state, one task in progress.
 5. Simple operations go to the cheap model.
 6. Search runs commands over live data, not a stale index.
 7. Permissions are deny-first; dangerous tools confirm before running.
 8. Steering uses markers, reminders and example pairs, not essays.
 9. Every loop is bounded: recursion_limit on every invoke.
10. The agent can run a check that proves its own work.
```

**Your working mode until the defense, borrowed from today.**

- verify-result loop: every change proved by a run, not by a feeling
- compression on, todo discipline on, one task per approach
- when a step fails twice, write it down and re-plan; do not push through

## Today, in one card

**The model is a few percent of the product; the harness around it is layered, deny-first, and cheap wherever it can be.**

**You can now defend:**
- Claude Code is one main loop over a flat message list, with five compression layers before every call and no planner model
- a `<system-reminder>` is injected per call and never persisted: the model sees what the history does not store
- deny rules override allow rules, and an OS sandbox holds even when the model is fooled

**In your repository:** `runs/session-12.md`, five techniques with file pointers, the two you transferred and why, both traces before and after.
**The trap of the day:** `SummarizationMiddleware` with a `fraction` trigger needs a model profile and raises `ValueError` without one; trigger by messages or tokens.
**Ask yourself:** the todo list is state rather than chat text: what does that buy the harness, and what does rewriting it whole avoid?
**Where this returns:** session 13 packages these pillars as `create_deep_agent`; session 15 files the permission model among the defenses.